# 03b — Northern IPCAs (northern BC + Yukon): connect the proposed areas

Crops the aligned stack to the **convex hull + 25 km** of the **13 proposed IPCAs north of 55°N** (northern BC + Yukon, 56–67°N), masks to that shape, **locks in both the existing PAs and the proposed IPCAs** (drafts treated as effectively protected), and **up-weights connectivity** (`transboundary_connectivity` + `climate_corridors` ×5) so the solve identifies the best land connecting them. Parameters live in `config.ANALYSES["northern_ipcas"]`; outputs → `output_data/iter6_northern_ipcas/`.

**Note:** the 13 anchors are selected from the 32-feature `proposed_pa_v2.shp` by `source_filter` (min_lat 55). Watch the "locked-in / budget" line in cell 3 — locked is ~41% of the window, so `budget_pct` must exceed that. **Kernel:** `R (y2y)`.

In [1]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "northern_ipcas"       # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=northern_ipcas)
prioritizr 8.1.0 | terra 1.9.34 | analysis=northern_ipcas | solver=highs (single solution)
objective=min_shortfall | budget=43% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=hull bounds=[-2206000, 2165000, -1575000, 3516000] (+mask) | lock_in=both
penalties: connectivity=0 | boundary=0 | neighbor=1e-05
outputs -> output_data/iter6_northern_ipcas


In [2]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ROI hull: cropped to 631 x 1351 cells + polygon mask
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 631 x 1351 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [3]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

Warning message:
“[vect] Z coordinates ignored”


planning units: 482,384 cells | budget = 43% = 207,425 cells
locked-in [existing PAs + lockin_northern_ipcas.gpkg]: 196,195 cells (40.7% of window) -- fits within budget


In [4]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight transboundary_connectivity x2.0 -> 2.0000
  up-weight climate_corridors x2.0 -> 2.0000


In [5]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

neighbor penalty ON (1e-05): binary rook adjacency derived from the PU raster
penalties -> connectivity=0 | boundary=0 | neighbor=1e-05  (0 = off)


In [6]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

Warning message in problem(x, zones(features, zone_names = names(x), feature_names = names(features)), :
“→ `features` has a layer with only zero values.”
A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (482384 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 2165000, -1575000, 3516000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 207425.1)
│├•penalties: 
││└•1:          neighbor penalties (`penalty` = 0.00001, …)
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 2)
│├•constraints: 
││└•1:          locked in constraints (196195 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      

In [7]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

LP has 1883971 rows; 1424393 cols; 9566745 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [1e-05, 2e+00]

  Bound   [1e+00, 1e+00]

  RHS     [1e+05, 2e+05]



Presolving model

1120129 rows, 852297 cols, 5673698 nonzeros 1s

1108041 rows, 840165 cols, 2502225 nonzeros 5s

Presolve reductions: rows 1108041(-775930); columns 840165(-584228); nonzeros 2502225(-7064520) 

Solving the presolved LP

IPX model has 1108041 rows, 840165 columns and 2502225 nonzeros

Input
    Number of variables:                                840165
    Number of free variables:                           0
    Number of constraints:                              1108041
    Number of equality constraints:                     0
    Number of matrix entries:                           2502225

    Matrix range:                                       [1e+00, 1e+00]

    RHS range:                                          [1e+04, 1e+04]

    Objective range:                                    [9e

In [8]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01   207425.1         43             11230


In [9]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter6_northern_ipcas/portfolio.tif
  output_data/iter6_northern_ipcas/selection_frequency.tif
  output_data/iter6_northern_ipcas/portfolio_representation.csv
  output_data/iter6_northern_ipcas/run_summary.json
